In [1]:
%pip install -q groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\mouza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key here and press Enter: ")
print("Key saved for this session. Length:", len(os.environ["GROQ_API_KEY"]))

Paste your Groq API key here and press Enter:  ········


Key saved for this session. Length: 56


In [3]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"   # open GPT-family model hosted on Groq

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
)
print(r.choices[0].message.content)

Hello!


In [4]:
import json

tool = {"type": "function", "function": {
    "name": "get_firm_history",
    "description": "Return every fiscal year on record for one firm, oldest first, up to through_fyear.",
    "parameters": {
        "type": "object",
        "properties": {
            "gvkey": {"type": "string", "description": "Six-character Compustat firm id, keep leading zeros, e.g. '001004'"},
            "through_fyear": {"type": "integer", "description": "Last fiscal year the agent may see"},
        },
        "required": ["gvkey"],
    },
}}

prompt = ("You are a risk analyst. Assess internal-control risk for the firm with gvkey 001004 "
          "in fiscal year 2018. Start by retrieving the firm's history.")

r = client.chat.completions.create(
    model=MODEL,
    tools=[tool],
    tool_choice="auto",
    messages=[{"role": "user", "content": prompt}],
)

msg = r.choices[0].message
if msg.tool_calls:
    call = msg.tool_calls[0].function
    args = json.loads(call.arguments)
    print("Tool called:", call.name)
    print("Arguments:", args)
    print("RESULT:", "PASS" if call.name == "get_firm_history" and str(args.get("gvkey")) == "001004" else "CHECK")
else:
    print("RESULT: CHECK - model answered without calling the tool")
    print(msg.content)

Tool called: get_firm_history
Arguments: {'gvkey': '001004', 'through_fyear': 2018}
RESULT: PASS
